# 02 · Datasets, batches and losses: what the optimiser sees

*Base Model Training*, notebook 2 of 4

A training run only ever sees what the data pipeline hands it. In this notebook you build that pipeline on top of the engine from notebook 1: split the rows by source, so that evaluation means what it claims; standardise features with statistics from the training rows only; cut shuffled mini-batches; and turn the network's scores into a loss with gradients. Two experiments then show what goes wrong when the split leaks and when the path from the loss to the weights is cut.

## What you will build

- `group_split(groups, fractions, rng)`: training, validation and test rows, with every source in exactly one partition;
- `fit_standardiser(X)`: column means and scales estimated from the training rows only;
- `shuffled_batches(n_rows, batch_size, rng)`: a mini-batch iterator that visits every training row once per epoch;
- `cross_entropy(logits, targets)`: the classification loss, on your engine;
- `squared_error_grad(w, x, y)`: one derivative by hand, checked by your engine and by central differences;
- 🧪 a row split against a group split, and 🧪 a cut gradient path.

## Learning objectives

By the end of this notebook you will be able to:

1. give training, validation and test rows their separate roles, and split by source when rows share one;
2. fit preprocessing on the training rows only and apply the same fixed transform everywhere else;
3. build shuffled training batches and deterministic evaluation batches, and average a loss correctly when batch sizes differ;
4. match the loss to the model's output, and connect a loss to its gradients with your engine;
5. recognise operations that cut the path from the loss to the weights.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you type a function yourself, then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for two plots, a scatter plot of the data and the learning curves. The numbers behind both plots are also printed. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the data are generated, and every random number comes from a seeded generator.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against its reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

As in notebook 1, `practice_via` checks a function through a **probe**: a small function that runs your code on a fresh example and returns plain numbers to compare. Here it lets a check pass a *seed* rather than a random generator, so that your function and the reference draw exactly the same random numbers.

In [ ]:
def practice_via(name, yours, reference, probe, cases, atol=1e-6):
    """Like practice, but compares probe(function, *case) for your function and for the reference.

    Use it when a function works by changing objects (filling in .grad, updating
    weights) or when several answers are correct: the probe runs the function on
    a fresh example and turns the outcome into plain numbers. Returns your
    function once the measurements match, otherwise the reference.
    """
    def measure_yours(*case):
        return probe(yours, *case)

    def measure_reference(*case):
        return probe(reference, *case)

    chosen = practice(name, measure_yours, measure_reference, cases, atol)
    return yours if chosen is measure_yours else reference

## 📦 Recap: what you built earlier

The hidden cell below defines the tiny tensor engine from notebook 1 under its usual names: `Tensor` with all its operations, `parameter`, `zeros_parameter`, `zero_grad`, `numerical_grad` and `relative_error`. It is the exact cell that later courses paste, so it also contains `layer_norm`, from notebook 1's exercises, and `cross_entropy`, which you write yourself in section 5. Run it; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 1 · The data: readings that come in groups

The generator below stands in for a common real situation: many rows come from the same **source**. Think of short clips cut from one recording, several visits by one patient, or many readings from one sensor. Here each source is a sensor that sits near its own operating point, so its readings cluster together.

Each reading has two raw measurements in very different units, one around 50 and one around 0.5, and a label: class 1 when the underlying signal lies in the top-right or bottom-left quadrant, the nonlinear rule from notebook 1. About 10% of the labels are flipped at random, so even a perfect model would score only about 90%.

`make_readings` returns `X`, the raw measurements of shape `(rows, 2)`; `y`, the integer labels of shape `(rows,)`; and `groups`, the source id of every row. The argument `flipped_fraction` makes some whole sources follow the opposite rule; it stays at 0 except in the first experiment.

In [ ]:
def make_readings(n_sources, rows_per_source, rng, spread=0.3, label_noise=0.1, flipped_fraction=0.0):
    """Generated sensor readings. Returns X (rows, 2) raw measurements, y (rows,) labels in {0, 1}
    and groups (rows,) source ids; the rows of one source cluster around that source's centre."""
    centres = rng.normal(size=(n_sources, 2))
    flipped = rng.random(n_sources) < flipped_fraction        # sources that follow the opposite rule
    groups = np.repeat(np.arange(n_sources), rows_per_source)
    signal = centres[groups] + spread * rng.normal(size=(groups.size, 2))
    noisy = rng.random(groups.size) < label_noise              # labels flipped at random
    y = ((signal[:, 0] * signal[:, 1] > 0) ^ flipped[groups] ^ noisy).astype(int)
    X = np.column_stack([50.0 + 10.0 * signal[:, 0], 0.5 + 0.1 * signal[:, 1]])
    return X, y, groups


X, y, groups = make_readings(160, 5, np.random.default_rng(0))
print("X", X.shape, X.dtype, "| y", y.shape, y.dtype, "| groups", groups.shape, "|", np.unique(groups).size, "sources")
print("first rows (measurement 1, measurement 2, label, source):")
print(np.column_stack([X[:6].round(2), y[:6], groups[:6]]))
print("class shares:", (np.bincount(y) / y.size).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for label, marker in [(0, "o"), (1, "^")]:
    ax.scatter(X[y == label, 0], X[y == label, 1], s=12, marker=marker, alpha=0.7, label=f"class {label}")
ax.set_xlabel("measurement 1 (raw units)")
ax.set_ylabel("measurement 2 (raw units)")
ax.set_title("800 readings from 160 sources")
ax.legend()
plt.show()

**What to notice.** Readings sit in small clusters, one per source. The classes follow the quadrants around the point (50, 0.5), with some labels on the "wrong" side: those are the flipped 10%. The two axes span very different ranges, about 20 to 80 against 0.2 to 0.8.

## 2 · Three roles, split by source

Each row gets exactly one role:

- **training** rows update the weights;
- **validation** rows guide decisions, such as which checkpoint to keep or which learning rate to use;
- **test** rows are used once, at the end, to measure the decision you made.

When rows share a source, split the **sources**, not the rows. If readings from one sensor land in both training and test, the test partly measures how well the model recognises that sensor, which says little about a sensor it has never seen. The same holds for patients, documents, recordings and customers, and, with time order, for days. A **split manifest** that records the seed and each partition's sources makes the split checkable later, even after the source column has been removed from the model's inputs.

In [ ]:
demo_order = np.random.default_rng(1).permutation(len(y))
row_train, row_test = demo_order[:480], demo_order[640:]
shared = np.intersect1d(groups[row_train], groups[row_test])
print(f"a random row split: {shared.size} of the {np.unique(groups[row_test]).size} test sources also appear in training")

### ✍️ Your turn: `group_split`

Write `group_split(groups, fractions, rng)`:

- **inputs:** `groups`, an integer array with one source id per row, shape `(rows,)`; `fractions`, a tuple such as `(0.6, 0.2, 0.2)` for training, validation and test; `rng`, a NumPy random generator;
- **output:** a tuple of three integer arrays `(train_rows, val_rows, test_rows)` of row indices, each in increasing order, such that all the rows of a source land in the same partition.

Follow these steps exactly, so that your split matches the reference for the same seed:

1. `sources = np.unique(groups)` gives the sorted ids; then call `shuffled = rng.permutation(sources)` once;
2. `n_train = round(fractions[0] * len(sources))` and `n_val = round(fractions[1] * len(sources))`; the test partition gets the remaining sources;
3. cut `shuffled` into three consecutive slices of sources, and turn each slice into row indices with `np.flatnonzero(np.isin(groups, slice_of_sources))`.

The probe passes a seed and builds a fresh generator from it, once for your function and once for the reference, so both draw the same permutation.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def group_split_reference(groups, fractions, rng):
    sources = np.unique(groups)
    shuffled = rng.permutation(sources)
    n_train = round(fractions[0] * len(sources))
    n_val = round(fractions[1] * len(sources))
    parts = shuffled[:n_train], shuffled[n_train:n_train + n_val], shuffled[n_train + n_val:]
    return tuple(np.flatnonzero(np.isin(groups, part)) for part in parts)

In [ ]:
def group_split(groups, fractions, rng):
    """Split rows into (train, val, test) row indices so that each source id lands in exactly one partition."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
def with_seed(split_fn, groups, fractions, seed):
    """Probe: call split_fn with a fresh generator made from seed."""
    return split_fn(groups, fractions, np.random.default_rng(seed))


group_split = practice_via("group_split", group_split, group_split_reference, with_seed, cases=[
    (groups, (0.6, 0.2, 0.2), 1),
    (np.array([3, 3, 1, 1, 7, 7, 7, 2]), (0.5, 0.25, 0.25), 2),     # four sources of uneven size
    (np.repeat(np.arange(10), 3), (0.8, 0.1, 0.1), 3),
    (np.array([5, 5, 5]), (1.0, 0.0, 0.0), 4),                       # edge case: one source, all of it training
])

In [ ]:
train_rows, val_rows, test_rows = group_split(groups, (0.6, 0.2, 0.2), np.random.default_rng(1))
for name, rows in [("training", train_rows), ("validation", val_rows), ("test", test_rows)]:
    print(f"{name:10} {rows.size:4} rows from {np.unique(groups[rows]).size:3} sources, class 1 share {y[rows].mean():.2f}")
train_sources, val_sources, test_sources = (set(groups[rows]) for rows in (train_rows, val_rows, test_rows))
assert not (train_sources & val_sources or train_sources & test_sources or val_sources & test_sources), \
    "a source appears in two partitions"
assert np.array_equal(np.sort(np.concatenate([train_rows, val_rows, test_rows])), np.arange(len(y))), \
    "every row needs exactly one role"
manifest = {"seed": 1, "fractions": (0.6, 0.2, 0.2), "test_sources": sorted(int(s) for s in test_sources)}
print("split manifest, first test sources:", manifest["test_sources"][:8], "...")

**Expected behaviour.** 480 training rows from 96 sources, and 160 validation and 160 test rows from 32 sources each, with class 1 shares near one half in every partition. Both assertions pass: no source appears in two partitions, and every row has exactly one role. The manifest records the seed and the test sources, which is enough to rebuild and audit this split later.

## 3 · Fit transforms on the training rows only

The two raw measurements live on very different scales: one around 50 with a spread of about 10, the other around 0.5 with a spread of about 0.1. Gradient descent struggles with that, because one learning rate cannot suit both directions. **Standardising** each column, $z = (x - \mu) / s$, puts both near mean 0 and spread 1.

The mean $\mu$ and the scale $s$ are *learned from data*, so they belong to training. Estimate them on the training rows, then apply the same fixed numbers to the validation and test rows, and later to every input the deployed model sees; saving them with the model is part of saving the model. Statistics computed from all the rows would let validation and test data shape the transform. For a mean and a scale the leak is small, but for transforms such as vocabularies or target encodings it can be large.

In [ ]:
print("raw training column means:", X[train_rows].mean(axis=0).round(3), "| spreads:", X[train_rows].std(axis=0).round(3))

### ✍️ Your turn: `fit_standardiser`

Write `fit_standardiser(X)`:

- **input:** `X`, the training rows, shape `(rows, features)`;
- **output:** a tuple `(mean, scale)` of arrays of shape `(features,)`: the column means and the column standard deviations (NumPy's default, `ddof=0`), with every scale raised to at least `1e-6`, so that a constant column cannot cause a division by zero.

Hint: `X.mean(axis=0)`, `X.std(axis=0)` and `np.maximum(..., 1e-6)`. Applying the transform is one line, `standardise(X, mean, scale)`, which is given below.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def fit_standardiser_reference(X):
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)

In [ ]:
def fit_standardiser(X):
    """Column means and standard deviations (at least 1e-6) of the training rows X, shape (rows, features)."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
fit_standardiser = practice("fit_standardiser", fit_standardiser, fit_standardiser_reference, cases=[
    (X[train_rows],),
    (np.random.default_rng(5).normal(3.0, 2.0, (50, 4)),),
    (np.array([[1.0, 10.0], [3.0, 10.0]]),),                 # edge case: a constant column
])


def standardise(X, mean, scale):
    return (X - mean) / scale


mean, scale = fit_standardiser(X[train_rows])
Z = standardise(X, mean, scale)
for name, rows in [("training", train_rows), ("validation", val_rows), ("test", test_rows)]:
    print(f"{name:10} column means {Z[rows].mean(axis=0).round(3)}, spreads {Z[rows].std(axis=0).round(3)}")

**Expected behaviour.** The training columns have mean 0 and spread 1, up to rounding. The validation and test columns come close, but not exactly, because their statistics never entered the fit: that small mismatch is what an honest transform looks like. For the constant column the scale is `1e-6`, and standardising it gives zeros instead of `nan`.

## 4 · Mini-batches

An optimiser rarely uses all the training rows for one update. It takes a **mini-batch**, averages the loss over it, and updates the weights; one pass over the training rows is an **epoch**. Two rules apply:

- **Training batches are shuffled**, with a new order every epoch, so that the model does not learn from an arbitrary file order, such as all of one source first or all of class 0 first.
- **Evaluation batches are deterministic.** They come in a fixed order and keep the final partial batch, so that every validation row counts exactly once and two checkpoints are compared on identical data.

When batches differ in size, average the loss over **rows**, not over batches. With 10 rows in batches of 4, 4 and 2, giving the last batch the same weight as the others would count each of its rows twice as much as the rest.

In [ ]:
batch_means, batch_sizes = np.array([0.5, 0.7, 2.0]), np.array([4, 4, 2])
print("mean of the batch means:", batch_means.mean().round(3),
      "| per-row mean:", ((batch_means * batch_sizes).sum() / batch_sizes.sum()).round(3))

### ✍️ Your turn: `shuffled_batches`

Write the generator function `shuffled_batches(n_rows, batch_size, rng)`:

- **inputs:** `n_rows`, the number of training rows; `batch_size`; `rng`, a NumPy generator;
- **yields:** integer arrays of positions `0 … n_rows - 1`, in the order given by one call to `rng.permutation(n_rows)`, cut into consecutive batches of `batch_size`; the last batch holds whatever is left.

Hints: draw `order = rng.permutation(n_rows)` once, then `for start in range(0, n_rows, batch_size):` and `yield order[start:start + batch_size]`. A function that contains `yield` returns an iterator: nothing runs until a loop asks for the next batch.

The probe turns your iterator into a list, using a fresh generator made from a seed.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def shuffled_batches_reference(n_rows, batch_size, rng):
    order = rng.permutation(n_rows)
    for start in range(0, n_rows, batch_size):
        yield order[start:start + batch_size]

In [ ]:
def shuffled_batches(n_rows, batch_size, rng):
    """Yield index arrays that cover range(n_rows) once, in a shuffled order, batch_size at a time."""
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
def batches_as_list(batches_fn, n_rows, batch_size, seed):
    """Probe: run the iterator with a fresh generator and collect every batch."""
    return list(batches_fn(n_rows, batch_size, np.random.default_rng(seed)))


shuffled_batches = practice_via("shuffled_batches", shuffled_batches, shuffled_batches_reference, batches_as_list, cases=[
    (10, 4, 0),      # batches of 4, 4 and 2
    (12, 4, 1),      # three full batches
    (3, 8, 2),       # edge case: a single, partial batch
])

In [ ]:
def ordered_batches(n_rows, batch_size):
    """Deterministic evaluation batches, final partial batch included."""
    for start in range(0, n_rows, batch_size):
        yield np.arange(start, min(start + batch_size, n_rows))


epoch_rng = np.random.default_rng(3)
for epoch in range(2):
    batches = list(shuffled_batches(train_rows.size, 128, epoch_rng))
    every_row_once = np.array_equal(np.sort(np.concatenate(batches)), np.arange(train_rows.size))
    print(f"epoch {epoch}: batch sizes {[b.size for b in batches]}, first rows {train_rows[batches[0][:4]]}, "
          f"every row once: {every_row_once}")
print("evaluation batches:", [b.size for b in ordered_batches(val_rows.size, 64)])

**Expected behaviour.** Both epochs produce batches of 128, 128, 128 and 96 rows, and every training row appears exactly once per epoch. The first rows differ between the epochs, because the generator moves on. The batches hold *positions* within `train_rows`, so a training loop reads `train_rows[batch]`. The validation rows always come in the same order, as batches of 64, 64 and 32.

## 5 · Match the loss to the output

The network produces one unnormalised score, or **logit**, per class. For mutually exclusive classes the **cross-entropy** loss turns each row into log-probabilities with a log-softmax and takes the negative log-probability of the true class, averaged over the rows:

$$L = -\frac{1}{n} \sum_{i=1}^{n} \log p_{i, y_i}, \qquad \log p_{i} = \operatorname{log\_softmax}(z_i).$$

It reads the integer class index $y_i$ rather than a one-hot vector, and it expects **logits**: applying a softmax first and then this loss would apply the softmax twice. Other outputs need other losses. Independent yes-or-no labels (multilabel classification) use one sigmoid and one binary cross-entropy per label. Regression uses squared error, which punishes large misses strongly, or absolute error, which is gentler with outliers.

Your engine already has `log_softmax` and indexing, so the loss takes three operations. Its gradient with respect to the logits is $(p - \text{onehot}(y))/n$, which your engine derives on its own.

In [ ]:
z_demo = Tensor([[2.0, 0.0], [0.0, 3.0]])
y_demo = np.array([0, 0])
log_p = z_demo.log_softmax(axis=-1)
print("log p:\n", log_p.data.round(4))
print("log p of the true classes:", log_p[np.arange(2), y_demo].data.round(4))

### ✍️ Your turn: `cross_entropy`

Write `cross_entropy(logits, targets)`:

- **inputs:** `logits`, a `Tensor` of shape `(n, classes)`; `targets`, an integer array of shape `(n,)`;
- **output:** a 0-d `Tensor` holding the mean over the rows of $-\log p_{i, y_i}$, built from your engine's operations so that `.backward()` works.

Hints: `logits.log_softmax(axis=-1)`, then index with `[np.arange(len(targets)), targets]`, then `-(...).mean()`.

The probe runs your loss on fresh logits, calls `backward()`, and compares both the loss and the gradient of the logits with the reference.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def cross_entropy_reference(logits, targets):
    log_probs = logits.log_softmax(axis=-1)
    return -log_probs[np.arange(len(targets)), targets].mean()

In [ ]:
def cross_entropy(logits, targets):
    """Mean cross-entropy of logits (n, classes) against integer targets (n,), as a 0-d Tensor."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
def loss_and_logit_grad(loss_fn, logits, targets):
    """Probe: compute the loss on fresh logits, backpropagate, and return (loss value, d loss / d logits)."""
    z = Tensor(np.array(logits, dtype=float), requires_grad=True)
    loss = loss_fn(z, targets)
    loss.backward()
    return float(loss.data), z.grad


ce_rng = np.random.default_rng(6)
cross_entropy = practice_via("cross_entropy", cross_entropy, cross_entropy_reference, loss_and_logit_grad, cases=[
    (z_demo.data, y_demo),
    (ce_rng.normal(size=(5, 3)), np.array([0, 2, 1, 1, 2])),
    (np.array([[1000.0, 0.0], [0.0, -1000.0]]), np.array([0, 1])),    # huge logits stay finite
    (ce_rng.normal(size=(1, 4)), np.array([3])),                        # edge case: a single row
])

In [ ]:
z_check = Tensor(ce_rng.normal(size=(4, 3)), requires_grad=True)
t_check = np.array([2, 0, 1, 2])
cross_entropy(z_check, t_check).backward()
p_check = np.exp(z_check.data - z_check.data.max(axis=1, keepdims=True))
p_check /= p_check.sum(axis=1, keepdims=True)
print("engine gradient equals (p - onehot) / n:", np.allclose(z_check.grad, (p_check - np.eye(3)[t_check]) / 4))
print(f"loss on the logits: {float(cross_entropy(Tensor(z_check.data), t_check).data):.3f}")
print(f"loss on probabilities, so the softmax runs twice: {float(cross_entropy(Tensor(p_check), t_check).data):.3f}")
print(f"even perfect probabilities cannot push it below {float(cross_entropy(Tensor(np.eye(3)[t_check]), t_check).data):.3f}")
print(f"huge logits stay finite: {float(cross_entropy(Tensor([[1000.0, 0.0], [0.0, -1000.0]]), np.array([0, 1])).data):.1f}")

**Expected behaviour.** For the two demo rows the losses are about 0.127 and 3.049, so the mean is about 1.588; the huge logits give a finite loss of exactly 500. The engine's gradient matches $(p - \text{onehot}(y))/n$. Feeding probabilities instead of logits gives a different, flatter loss that cannot fall below about 0.551 for three classes, even when the probabilities are perfect: training on it stalls.

## 6 · One derivative by hand, checked twice

The lesson's worked example uses a one-weight model, $\hat y = wx$, with the squared-error loss $L = (wx - y)^2$. The chain rule gives

$$\frac{\partial L}{\partial w} = 2(wx - y)\,x .$$

With $x = 2$, $w = 1$ and $y = 3$, the prediction is 2, the error is $-1$, and the gradient is $2 \times (-1) \times 2 = -4$. A gradient-descent step with learning rate 0.1 moves $w$ to $1 - 0.1 \times (-4) = 1.4$, and the prediction to 2.8, closer to 3. The gradient says in which direction, and how steeply, the loss changes *here*; it promises nothing about a large step.

### ✍️ Your turn: `squared_error_grad`

Write `squared_error_grad(w, x, y)`:

- **inputs:** numbers, or arrays of the same shape, `w`, `x` and `y`;
- **output:** $2(wx - y)x$, elementwise.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def squared_error_grad_reference(w, x, y):
    return 2 * (w * x - y) * x

In [ ]:
def squared_error_grad(w, x, y):
    """d/dw of (w * x - y) ** 2, elementwise."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
squared_error_grad = practice("squared_error_grad", squared_error_grad, squared_error_grad_reference, cases=[
    (1.0, 2.0, 3.0),                                                    # the worked example: -4
    (np.array([0.5, -1.0]), np.array([1.0, 3.0]), np.array([2.0, 0.0])),
    (2.0, 0.0, 5.0),                                                    # edge case: x = 0, so w has no effect
])

In [ ]:
w = Tensor(1.0, requires_grad=True)


def squared_error():
    return (w * 2.0 - 3.0) ** 2


squared_error().backward()
numerical = numerical_grad(squared_error, w)
print(f"by hand {squared_error_grad(1.0, 2.0, 3.0)}, engine {float(w.grad)}, central difference {float(numerical):.6f}")
w.data -= 0.1 * w.grad
print(f"after one step with learning rate 0.1: w = {float(w.data):.1f}, prediction = {2 * float(w.data):.1f}")
assert np.isclose(w.grad, -4.0) and np.isclose(numerical, -4.0) and np.isclose(w.data, 1.4)

## 7 · 🧪 Experiments

### 🧪 Experiment 1: how much does a row split flatter the score?

A **1-nearest-neighbour** classifier predicts the label of the closest training row. It is a pure memoriser, which makes it a sharp probe for leakage. In this version of the data a quarter of the sources follow the opposite rule (`flipped_fraction=0.25`), something no model can know about a source it has never seen, and each source's readings sit very close together (`spread=0.05`), like near-identical clips from one recording.

With a **row** split, most test rows have a sibling from the same source in training, so the memoriser copies that source's label and looks excellent. With a **group** split it must generalise to new sources, and the flipped ones defeat it. Only the second number describes performance on new sources. Change `flipped_fraction` to 0.0 and the gap closes: leakage hurts most when sources differ in ways that the inputs cannot explain.

In [ ]:
flipped_fraction = 0.25  # 🧪 try 0.0, then 0.5
spread = 0.05            # 🧪 try 0.3, so that readings from one source spread out


def nearest_neighbour_accuracy(Z_train, y_train, Z_test, y_test):
    distances = ((Z_test[:, None, :] - Z_train[None, :, :]) ** 2).sum(axis=-1)   # (test rows, training rows)
    return (y_train[distances.argmin(axis=1)] == y_test).mean()


X_leak, y_leak, groups_leak = make_readings(160, 5, np.random.default_rng(20), spread=spread,
                                            label_noise=0.0, flipped_fraction=flipped_fraction)
order = np.random.default_rng(21).permutation(len(y_leak))
splits = {"row split": (order[:480], order[640:])}
group_train, _, group_test = group_split(groups_leak, (0.6, 0.2, 0.2), np.random.default_rng(21))
splits["group split"] = (group_train, group_test)
for name, (fit_rows, eval_rows) in splits.items():
    m, s = fit_standardiser(X_leak[fit_rows])
    accuracy = nearest_neighbour_accuracy(standardise(X_leak[fit_rows], m, s), y_leak[fit_rows],
                                          standardise(X_leak[eval_rows], m, s), y_leak[eval_rows])
    print(f"{name:11}: 1-nearest-neighbour test accuracy {accuracy:.1%}")

**What to notice.** With the default settings the row split reports a much higher accuracy than the group split, although the model and the data are identical: the row split rewards recognising sources. Neither number is wrong, but they answer different questions, and only the group split answers "how well does this work on a sensor we have not seen?".

### 🧪 Experiment 2: cut the path from the loss to the weights

Training works only if every step from the weights to the loss is recorded. Three common ways to cut that path are:

- **detaching** the logits with `Tensor(logits.data)`, a new tensor with the same numbers and no history (frameworks such as PyTorch call this `.detach()`);
- turning the loss into a plain Python number, `float(loss.data)`, before calling `backward`, which a number does not have;
- running the forward pass in a mode that records nothing, such as `torch.no_grad()` in PyTorch.

The cell trains the same small network twice for 30 steps, once with the cut you choose and once normally, and reports how far the weights moved. Logging `float(loss.data)` *after* `backward()` is harmless; it is a problem only when the number replaces the tensor you differentiate.

In [ ]:
cut = "detach"  # 🧪 try "python float" or "none"


def train_briefly(cut, steps=30, learning_rate=0.1):
    init = np.random.default_rng(30)
    weights = [parameter(2, 8, rng=init), zeros_parameter(8), parameter(8, 2, rng=init), zeros_parameter(2)]
    start = [w.data.copy() for w in weights]
    batch_rng = np.random.default_rng(31)
    for _ in range(steps):
        rows = train_rows[batch_rng.choice(train_rows.size, 32, replace=False)]
        logits = (Tensor(Z[rows]) @ weights[0] + weights[1]).relu() @ weights[2] + weights[3]
        if cut == "detach":
            logits = Tensor(logits.data)            # the same numbers, no history
        loss = cross_entropy(logits, y[rows])
        if cut == "python float":
            loss = float(loss.data)                 # a plain number: it has no backward()
        zero_grad(weights)
        loss.backward()
        for w in weights:
            if w.grad is not None:                  # no gradient reached this weight
                w.data -= learning_rate * w.grad
    return sum(float(np.abs(w.data - s).sum()) for w, s in zip(weights, start))


try:
    print(f"cut = {cut!r}: the weights moved by {train_briefly(cut):.4f} in total")
except AttributeError as error:
    print(f"cut = {cut!r} fails loudly: {error}")
print(f"cut = 'none', for comparison: the weights moved by {train_briefly('none'):.4f} in total")

**What to notice.** A detached forward pass fails *silently*: `backward()` runs, no gradient reaches the weights, and they do not move at all. The Python float fails *loudly*, with an `AttributeError`. The silent failure is the dangerous one, which is why the next notebook checks that the weights actually change after an update.

## 8 · Put it together: a complete data pipeline

Everything in this notebook now joins up. The split is by source and the standardiser was fitted on the training rows. A 2 → 16 → 2 network trains for 40 epochs of shuffled mini-batches of 32 with plain gradient descent, and after every epoch it is evaluated on the validation rows in deterministic batches, with the loss averaged over rows. The test rows stay untouched; notebook 4 uses them once, at the very end.

In [ ]:
def network(weights, inputs):
    return (Tensor(inputs) @ weights[0] + weights[1]).relu() @ weights[2] + weights[3]


def evaluate(weights, rows, batch_size=64):
    """Per-row mean loss and accuracy over `rows`, in deterministic batches."""
    total_loss, correct = 0.0, 0
    for batch in ordered_batches(rows.size, batch_size):
        picked = rows[batch]
        logits = network(weights, Z[picked])
        total_loss += float(cross_entropy(logits, y[picked]).data) * picked.size
        correct += int((logits.data.argmax(axis=1) == y[picked]).sum())
    return total_loss / rows.size, correct / rows.size


init_rng = np.random.default_rng(40)
weights = [parameter(2, 16, rng=init_rng), zeros_parameter(16), parameter(16, 2, rng=init_rng), zeros_parameter(2)]
batch_rng = np.random.default_rng(41)
learning_rate, epochs = 0.1, 40
history = {"train": [], "val": [], "val_accuracy": []}
start_val_loss, start_val_accuracy = evaluate(weights, val_rows)
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch in shuffled_batches(train_rows.size, 32, batch_rng):
        rows = train_rows[batch]
        loss = cross_entropy(network(weights, Z[rows]), y[rows])
        zero_grad(weights)
        loss.backward()
        for w in weights:
            w.data -= learning_rate * w.grad
        epoch_loss += float(loss.data) * rows.size
    val_loss, val_accuracy = evaluate(weights, val_rows)
    history["train"].append(epoch_loss / train_rows.size)
    history["val"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

majority_class = np.bincount(y[train_rows]).argmax()
baseline_accuracy = (y[val_rows] == majority_class).mean()
for epoch in (1, 2, 5, 10, 20, 40):
    print(f"epoch {epoch:2}: training loss {history['train'][epoch - 1]:.3f}, validation loss {history['val'][epoch - 1]:.3f}, "
          f"validation accuracy {history['val_accuracy'][epoch - 1]:.1%}")
print(f"validation loss {start_val_loss:.3f} before training -> {history['val'][-1]:.3f} after {epochs} epochs")
print(f"validation accuracy {start_val_accuracy:.1%} -> {history['val_accuracy'][-1]:.1%}; "
      f"always predicting the majority training class scores {baseline_accuracy:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
epoch_numbers = np.arange(1, epochs + 1)
ax.plot(epoch_numbers, history["train"], label="training loss (per row)")
ax.plot(epoch_numbers, history["val"], label="validation loss (per row)")
ax.axhline(np.log(2), linestyle=":", color="grey", label="log 2: guessing 50/50")
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy")
ax.set_title("Learning curves")
ax.legend()
plt.show()

In [ ]:
assert not (set(groups[train_rows]) & (set(groups[val_rows]) | set(groups[test_rows]))), "split by source"
assert np.allclose(Z[train_rows].mean(axis=0), 0.0, atol=1e-9), "the standardiser was fitted on the training rows"
assert history["val"][-1] < start_val_loss - 0.1, "the validation loss should fall clearly"
assert history["val"][-1] < np.log(2), "the model should beat 50/50 guessing on validation"
assert history["val_accuracy"][-1] >= baseline_accuracy + 0.2, "accuracy should beat the majority baseline by a wide margin"
print("All checks passed.")

**Expected behaviour.** Before training, the randomly initialised network scores about 24% on validation, worse than guessing, with a loss of about 1.06. After one epoch the validation loss is about 0.6, and it keeps falling, more slowly, to about 0.38 after 40 epochs, while validation accuracy settles near 91%, about 40 percentage points above the majority baseline of 51%. The ceiling comes from the 10% of flipped labels, not from the network: no model can predict a coin flip. Here the training loss even stays slightly above the validation loss. Two effects contribute: it is averaged during each epoch, while the weights are still improving, and which rows carry flipped labels varies by chance between the partitions.

## Exercises

### Exercise 1: a masked loss, `cross_entropy_masked`

Sequence models pad short sequences to a common length, and padded positions must not count. Write `cross_entropy_masked(logits, targets, mask)`, where `mask` is a float array of shape `(n,)` holding 1.0 for rows that count and 0.0 for rows that do not. Return the sum of $-\log p_{i, y_i}$ over the kept rows divided by the number of kept rows, using `max(mask.sum(), 1.0)` so that an all-zero mask cannot divide by zero. The engine's own `cross_entropy` in the recap cell supports exactly this through its `mask` argument.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def cross_entropy_masked_reference(logits, targets, mask):
    picked = logits.log_softmax(axis=-1)[np.arange(len(targets)), targets]
    mask = np.asarray(mask, dtype=np.float64)
    return -(picked * mask).sum() / max(mask.sum(), 1.0)

In [ ]:
def cross_entropy_masked(logits, targets, mask):
    """Mean cross-entropy over the rows where mask is 1.0."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
def masked_loss_and_grad(loss_fn, logits, targets, mask):
    """Probe: like loss_and_logit_grad, with a mask."""
    z = Tensor(np.array(logits, dtype=float), requires_grad=True)
    loss = loss_fn(z, targets, mask)
    loss.backward()
    return float(loss.data), z.grad


mask_rng = np.random.default_rng(7)
cross_entropy_masked = practice_via("cross_entropy_masked", cross_entropy_masked, cross_entropy_masked_reference,
                                    masked_loss_and_grad, cases=[
    (mask_rng.normal(size=(4, 3)), np.array([0, 1, 2, 1]), np.array([1.0, 1.0, 0.0, 1.0])),
    (mask_rng.normal(size=(3, 2)), np.array([1, 0, 1]), np.ones(3)),         # all kept: the ordinary mean
    (mask_rng.normal(size=(2, 2)), np.array([0, 1]), np.zeros(2)),           # edge case: nothing kept
])

### Exercise 2: split by time, `time_split`

When rows are ordered in time, the future must not train a model that is then evaluated on the past. Write `time_split(times, fractions)`: sort the row indices with `np.argsort(times, kind="stable")` (stable, so that ties keep their original order), give the earliest `round(fractions[0] * n)` rows to training and the next `round(fractions[1] * n)` rows to validation, give the rest to test, and return the three index arrays, each sorted in increasing order.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def time_split_reference(times, fractions):
    order = np.argsort(times, kind="stable")
    n_train, n_val = round(fractions[0] * len(times)), round(fractions[1] * len(times))
    return tuple(np.sort(part) for part in (order[:n_train], order[n_train:n_train + n_val], order[n_train + n_val:]))

In [ ]:
def time_split(times, fractions):
    """Row indices (train, val, test): the earliest rows train, the latest rows test."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
time_split = practice("time_split", time_split, time_split_reference, cases=[
    (np.array([5.0, 1.0, 4.0, 2.0, 3.0]), (0.6, 0.2, 0.2)),
    (np.arange(10.0)[::-1], (0.5, 0.3, 0.2)),
    (np.array([2.0, 1.0, 1.0, 2.0]), (0.5, 0.25, 0.25)),     # edge case: tied times keep their order
])

### Exercise 3: an open question

A shop has several rows per customer, and the same customers appear again next month. You want to predict next month's purchases. Which split unit would you choose, customer or time or both, and what exactly would each choice measure? Write two or three sentences, and say which choice matches the way the model will be used.

## Common errors and debugging

- **Rows from one source in two partitions.** Split the source ids, then map them back to rows; assert that the partitions share no source.
- **A transform fitted on all the rows.** Fit means, scales and vocabularies on the training rows only, and save them with the model.
- **Shuffled or truncated validation batches.** Evaluation needs a fixed order and every row; drop nothing.
- **Averaging batch means when batch sizes differ.** Weight each batch by its number of rows, or sum the losses and divide by the total.
- **A softmax before cross-entropy.** The loss expects logits; probabilities give a flatter loss that cannot reach zero.
- **Float targets.** Class targets are integer indices; a float index raises an `IndexError`.
- **`'float' object has no attribute 'backward'`.** The loss was turned into a number before `backward()`. Keep the tensor and log `float(loss.data)` afterwards.
- **The weights never move, and nothing complains.** Something detached the graph; check that `.grad` is not `None` after `backward()`.

## Summary

- Training, validation and test rows have different jobs; when rows share a source, the split unit is the source.
- Transforms are learned on the training rows only and applied unchanged everywhere else.
- Training batches are shuffled every epoch; evaluation batches are fixed and complete, and losses are averaged over rows.
- Cross-entropy takes logits and integer targets; your engine derives its gradient, $(p - \text{onehot}(y))/n$, without help.
- Detaching tensors, converting the loss to a number, or recording nothing in the forward pass cuts the path from the loss to the weights; the detached case fails silently.

## Next

Notebook 3 wraps the weights into modules with `parameters()`, adds dropout and the Adam optimiser, diagnoses a failing training loop, and saves checkpoints that resume training exactly.